## Ingest Bronze-Products Table

In [0]:
%run "../shared/00.common-variables"

In [0]:
%run "../shared/01.schemas"

In [0]:
%run "../shared/02.helper-functions"

In [0]:
from datetime import datetime
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DateType, TimestampType
from datetime import datetime
from pyspark.sql import functions as F
from zoneinfo import ZoneInfo
from pyspark.sql import Window



**Keeping all fields, nullable= True**

Because later, when discussing architecture you can say:

Bronze prioritizes ingestion reliability over business correctness.
Hence, All columns are nullable.

And, Business Data Quality Check is applied in Silver.

## Transformation and Standardization Pipeline

In [0]:
# %sql
# -- FOR TESTING PURPOSES ONLY: Comment after running
# TRUNCATE TABLE olist.bronze.customers;
# TRUNCATE TABLE olist.logs.pipeline_logs;

In [0]:
bronze_products_df = spark.read.format('delta') \
                               .table(f"{catalog_name}.{bronze_schema}.products")

display(bronze_products_df)

In [0]:
bronze_products_df.printSchema()

In [0]:
bronze_products_df = bronze_products_df.withColumnsRenamed(
    {
        "product_name_lenght":"product_name_length",
        "product_description_lenght":"product_description_length"
    }
)

In [0]:

ingestion_errors_df = spark.sql("""
    select * from olist.logs.pipeline_logs
    where pipeline_name = 'bronze_products'
""")

display(ingestion_errors_df)

In [0]:
display(bronze_products_df.summary())

In [0]:
dq_metrics = (
    bronze_products_df.agg(
        F.count("*").alias("total_records"),
        F.countDistinct("product_id").alias("distinct_product_ids"),
        F.countDistinct("product_category_name").alias("distinct_product_categories"),

        F.sum(F.when(F.col("product_name_length") < 0, 1).otherwise(0)).alias("product_name_length_negative"),

        F.sum(F.when(F.col("product_description_length") < 0, 1).otherwise(0)).alias("product_description_length_negative"),

        F.sum(F.when(F.col("product_photos_qty") < 0, 1).otherwise(0)).alias("product_photos_qty_negative"),

        F.sum(F.when(F.col("product_weight_g") < 0, 1).otherwise(0)).alias("product_weight_g_negative"),

        F.sum(F.when(F.col("product_length_cm") < 0, 1).otherwise(0)).alias("product_length_cm_negative"),

        F.sum(F.when(F.col("product_height_cm") < 0, 1).otherwise(0)).alias("product_height_cm_negative"),

        F.sum(F.when(F.col("product_width_cm") < 0, 1).otherwise(0)).alias("product_width_cm_negative"),

        F.sum(
            F.when(
                (F.col("product_category_name").isNull()) |
                (F.trim(F.col("product_category_name")) == ""),
                1
            ).otherwise(0)
        ).alias("product_category_name_nulls_blanks")
    )
)

display(dq_metrics)

In [0]:
# Replace invalid numeric values with NULL

# Business Rule:
    # Numeric attributes representing counts, lengths or weights cannot be negative. 
    # Any negative value is considered invalid and is replaced with NULL instead of dropping the entire record.

numeric_columns = [
    "product_name_length",
    "product_description_length",
    "product_photos_qty",
    "product_weight_g",
    "product_length_cm",
    "product_height_cm",
    "product_width_cm"
]

for col_name in numeric_columns:
    bronze_products_df = bronze_products_df.withColumn(
        col_name,
        F.when(F.col(col_name) < 0, None)
         .otherwise(F.col(col_name))
    )

In [0]:
# Remove trailing white spaces from product categories
# --------- AND -----------------------------------
# Convert all product category names to lowercase

silver_products = (
    bronze_products_df
    .withColumn("product_category_name",F.trim(F.col("product_category_name")))
    .withColumn("product_category_name",F.lower(F.col("product_category_name")))
)



In [0]:
from pyspark.sql import Window

# There are rows where cateogory is null, name length is null, or product weight or length are null
# Keeping them for now.

# Filter out Records with NULL product_id
 
silver_products = silver_products.filter(F.col("product_id").isNotNull())

# Filter out Duplicate product_id

# 1. Define the window partitioned by product and ordered by timestamp descending
window_prod_id = Window.partitionBy("product_id").orderBy(F.col("ingestion_timestamp").desc())

# 2. Filter for the top row (row_number == 1) and drop the temporary column
silver_products = silver_products.withColumn("row_num", F.row_number().over(window_prod_id)) \
               .filter(F.col("row_num") == 1) \
               .drop("row_num")

## For simplicity, not putting the code to remove records where integer values that should be POSITIVE are NEGATIVE.


In [0]:
# Bring Translated product category names in English

# 1. Remove corrupt records (if any) from "translations" table
product_category_name_translation_df = spark.read.table(f"{catalog_name}.{bronze_schema}.product_category_name_translation")
product_category_name_translation_df = product_category_name_translation_df.filter(F.col("_corrupt_record").isNull())\
.select(F.col("product_category_name"),F.col("product_category_name_english"))

# 2. Join with Bronze Products to bring in the English Names to our dataframe
silver_products = silver_products.join(product_category_name_translation_df, on="product_category_name", how="left")\
    .select("product_id","product_category_name","product_category_name_english","product_name_length","product_description_length","product_photos_qty","product_weight_g","product_length_cm","product_height_cm","product_width_cm","ingestion_timestamp","pipeline_run_id"
)

# display(silver_products.columns)


In [0]:
silver_products.createOrReplaceTempView("silver_products_source_updates")

In [0]:
%sql
MERGE INTO olist.silver.products as target
USING silver_products_source_updates as source
on target.product_id = source.product_id
WHEN MATCHED THEN
UPDATE SET
    target.product_id = source.product_id,
    target.product_category_name = source.product_category_name,
    target.product_category_name_english = source.product_category_name_english,
    target.product_name_length = source.product_name_length,
    target.product_description_length = source.product_description_length,
    target.product_photos_qty = source.product_photos_qty,
    target.product_weight_g = source.product_weight_g,
    target.product_length_cm = source.product_length_cm,
    target.product_height_cm = source.product_height_cm,
    target.product_width_cm = source.product_width_cm,
    target.ingestion_timestamp = source.ingestion_timestamp,
    target.pipeline_run_id = source.pipeline_run_id
WHEN NOT MATCHED THEN
INSERT
    (   
        product_id,
        product_category_name,
        product_category_name_english,
        product_name_length,
        product_description_length,
        product_photos_qty,
        product_weight_g,
        product_length_cm,
        product_height_cm,
        product_width_cm,
        ingestion_timestamp,
        pipeline_run_id
    )
VALUES
    (
        source.product_id,
        source.product_category_name,
        source.product_category_name_english,
        source.product_name_length,
        source.product_description_length,
        source.product_photos_qty,
        source.product_weight_g,
        source.product_length_cm,
        source.product_height_cm,
        source.product_width_cm,
        source.ingestion_timestamp,
        source.pipeline_run_id
    )



In [0]:
#

In [0]:
## ---------------------------Config Variables-------------------------------------------------------------------
start_time = datetime.now(ZoneInfo("Asia/Kolkata"))
end_time = None

table_name = "products"
business_key = "product_id"
full_table_name = f"{catalog_name}.{bronze_schema}.{table_name}"
source_file_path = f"{landing_folder_path}/olist_products_dataset.csv"
file_name = 'olist_products_dataset.csv'

pipeline_name = f"{bronze_schema}_{table_name}"
pipeline_run_id = generate_pipeline_run_id(pipeline_name, start_time)


## ---------------------------Pipeline Run-------------------------------------------------------------------


source_row_count = None
target_row_count = None
null_business_key_count = None
duplicated_business_key_count = None
corrupt_row_count = None
status = None
error_message = None

pipeline_exception = None

#----------- Pipeline Control Variables ----------#

file_checksum = None
retry_count = 0
file_location = source_file_path

try:

    file_checksum = generate_file_checksum(source_file_path)

    if is_file_already_processed(file_checksum, full_pipeline_control_table_name):
        status = "SKIPPED_DUPLICATE"
        error_message = "File already processed"

        # Move file to duplicates/ folder
        target_file_path = f"{duplicates_folder_path}/{file_name}"
        dbutils.fs.mv(source_file_path, target_file_path)
        file_location = target_file_path

    else:
    
        products_df = spark.read.format('csv')\
                        .option("header",True)\
                        .option("mode","PERMISSIVE")\
                        .option("columnNameOfCorruptRecord", "_corrupt_record")\
                        .schema(products_schema)\
                        .load(source_file_path)


        products_df = add_ingestion_metadata(products_df, pipeline_run_id)

        source_row_count, target_row_count, null_order_id_count,\
        duplicated_order_id_count, corrupt_row_count = calculate_dq_metrics(products_df,business_key)

        # target_row_count = source_row_count 
        # Because no transformation, deduplication etc. being done on source data. and writing count() again would trigger an action which could be expensive with large data

        table_location = f"abfss://olist@olistretailplatform.dfs.core.windows.net/bronze/{table_name}"

        products_df.write \
            .format("delta") \
            .mode("append") \
            .saveAsTable(full_table_name)


        status = "SUCCESS"
        error_message = None
    
        # Move file to processed/ folder
        target_file_path = f"{processed_folder_path}/{file_name}"
        dbutils.fs.mv(source_file_path, target_file_path)
        file_location = target_file_path
            

except Exception as e:
    pipeline_exception = e
    error_message = str(e)
    status = "FAILED"

    # Move file to failed/ folder
    target_file_path = f"{failed_folder_path}/{file_name}"
    dbutils.fs.mv(source_file_path, target_file_path)
    file_location = target_file_path
    

finally:
    end_time = datetime.now(ZoneInfo(my_zone))
    try:

        pipeline_control_data = [(
        file_checksum,
        file_name,
        source_file_path,
        pipeline_name,
        pipeline_run_id,
        table_name,
        status,
        retry_count,
        start_time,
        end_time,
        source_row_count,
        target_row_count,
        error_message,
        file_location
        )]

        pipeline_control_df = spark.createDataFrame(
            pipeline_control_data,
            schema=pipeline_control_schema
        )

        update_pipeline_control(pipeline_control_df, full_pipeline_control_table_name)
        
        logs_data = [
        (
            pipeline_run_id,
            pipeline_name,
            table_name,
            start_time,
            end_time,
            status,
            source_row_count,
            target_row_count,
            null_business_key_count,
            duplicated_business_key_count,
            corrupt_row_count,
            error_message
        )
        ]

        logs_df = spark.createDataFrame(
            logs_data,
            schema=logs_df_schema
        )

        write_logs(logs_df, full_logs_table_name)
        
        if pipeline_exception:
            raise pipeline_exception


    except Exception as logging_error:
        print(f"logging FAILED: {logging_error}")
        raise # This will make the notebook fail. If we don't use this, the message is just printed and notebook run continues.Which we don't want.



In [0]:
%sql
select * from olist.bronze.products


In [0]:
%sql
select * from olist.logs.pipeline_logs

## For Reference

### Preparing Dataframe : current-table_df

In [0]:


# orders_df = spark.read.format('csv')\
#                  .option("header",True)\
#                  .option("mode","PERMISSIVE")\
#                  .option("columnNameOfCorruptRecord", "_corrupt_record")\
#                  .schema(orders_schema)\
#                  .load(source_file_path)


# orders_df = add_ingestion_metadata(orders_df, pipeline_run_id)





In [0]:
# display(orders_df.limit(20))

In [0]:
# orders_df.printSchema()

In [0]:
# source_row_count, target_row_count, null_order_id_count,\
#     duplicated_order_id_count, corrupt_row_count = calculate_dq_metrics(orders_df,"order_id")

# # target_row_count = source_row_count 
# # Because no transformation, deduplication etc. being done on source data. and writing count() again would trigger an action which could be expensive with large data

# end_time = datetime.now(ZoneInfo(my_zone))
# status = "SUCCESS"
# error_message = None

### Create Table: current_table

In [0]:
# spark.sql(f"""CREATE TABLE IF NOT EXISTS {full_table_name} 
# (
#     order_id                         STRING,
#     order_status                     STRING,
#     customer_id                      STRING,
#     order_purchase_timestamp         TIMESTAMP,
#     order_approved_at                TIMESTAMP,
#     order_delivered_carrier_date     TIMESTAMP,
#     order_delivered_customer_date    TIMESTAMP,
#     order_estimated_delivery_date    DATE,
#     _corrupt_record                  STRING,
#     ingestion_timestamp              TIMESTAMP NOT NULL,
#     source_file_name                 STRING NOT NULL,
#     source_system                    STRING NOT NULL,
#     pipeline_run_id                  STRING NOT NULL
# )
# USING DELTA
# LOCATION "abfss://olist@olistretailplatform.dfs.core.windows.net/bronze/orders"
# """)

In [0]:
# orders_df.write \
#     .format("delta") \
#     .mode("append") \
#     .saveAsTable(full_table_name)

### Logging

In [0]:
# logs_data = [
#     (
#         pipeline_run_id,
#         pipeline_name,
#         table_name,
#         start_time,
#         end_time,
#         status,
#         source_row_count,
#         target_row_count,
#         null_order_id_count,
#         duplicated_order_id_count,
#         corrupt_row_count,
#         error_message
#     )
# ]

# logs_df = spark.createDataFrame(
#     logs_data,
#     schema=logs_df_schema
# )

In [0]:
# display(logs_df)

In [0]:
# write_logs(logs_df, full_logs_table_name)